In [12]:
import os

import chromadb
from dotenv import load_dotenv
from google import genai

load_dotenv()
assert os.getenv("GEMINI_API_KEY"), "Add GEMINI_API_KEY to your .env file"

client = genai.Client()

# Use the SAME embedding model as store.ipynb. Model names change over time, so check Google's docs if one fails.
EMBED_MODEL = "gemini-embedding-001"
GEN_MODEL = "gemini-3.5-flash-lite"      # was "gemini-2.5-flash"
DB_PATH = "chroma_db"
COLLECTION_NAME = "notes"

chroma = chromadb.PersistentClient(path=DB_PATH)
collection = chroma.get_collection(COLLECTION_NAME)
assert collection.count() > 0, "The collection is empty. Run store.ipynb first."
print("loaded", collection.count(), "chunks")

loaded 9 chunks


In [13]:
def embed(texts):
    result = client.models.embed_content(model=EMBED_MODEL, contents=texts)
    return [e.values for e in result.embeddings]

In [14]:
def retrieve(question, k=3):
    """Return the k chunks closest in meaning to the question."""
    q_vec = embed([question])[0]
    res = collection.query(query_embeddings=[q_vec], n_results=min(k, collection.count()))
    hits = []
    for text, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({
            "text": text,
            "source": meta["source"],
            "chunk_no": meta["chunk_no"],
            "distance": dist,        # smaller = closer
        })
    return hits


for h in retrieve("What happens when processes wait forever?"):
    print(f"{h['source']} chunk {h['chunk_no']}  distance {h['distance']:.3f}")
    print("  ", h["text"][:80], "...")

os.txt chunk 1  distance 0.445
   A deadlock happens when two or more processes wait for each other forever. Proce ...
os.txt chunk 2  distance 0.756
   A process is a running program with its own memory space. A thread is a smaller  ...
dbms.txt chunk 3  distance 0.914
   ACID describes the four properties of a reliable transaction: Atomicity, Consist ...


In [15]:
def answer(question, k=3):
    hits = retrieve(question, k)
    context = "\n\n".join(
        f"[{i}] ({h['source']}, chunk {h['chunk_no']})\n{h['text']}"
        for i, h in enumerate(hits, start=1)
    )
    prompt = f"""You answer questions using ONLY the notes below.
- Cite the notes you used like [1] or [2].
- If the notes do not contain the answer, reply with exactly: NOT_FOUND

NOTES:
{context}

QUESTION: {question}
ANSWER:"""
    response = client.models.generate_content(model=GEN_MODEL, contents=prompt)
    text = (response.text or "").strip()
    found = "NOT_FOUND" not in text
    return {
        "answer": text if found else "I couldn't find this in your notes.",
        "found": found,
        "sources": hits if found else [],
    }


def ask(question):
    r = answer(question)
    print("Q:", question)
    print("A:", r["answer"])
    for h in r["sources"]:
        print(f"   source: {h['source']} chunk {h['chunk_no']} (distance {h['distance']:.3f})")
    print()

In [16]:
ask("What is a deadlock?")
ask("Which join keeps all rows from the left table?")
ask("Who won the cricket world cup?")

Q: What is a deadlock?
A: A deadlock happens when two or more processes wait for each other forever [1].
   source: os.txt chunk 1 (distance 0.513)
   source: dbms.txt chunk 3 (distance 0.848)
   source: os.txt chunk 2 (distance 0.861)

Q: Which join keeps all rows from the left table?
A: A left join keeps all rows from the left table [1].
   source: dbms.txt chunk 2 (distance 0.354)
   source: dbms.txt chunk 1 (distance 0.817)
   source: dbms.txt chunk 3 (distance 0.910)

Q: Who won the cricket world cup?
A: I couldn't find this in your notes.



In [17]:
TEST_SET = [
    ("What is a deadlock?", "os.txt"),
    ("What happens when two processes wait forever for each other's resources?", "os.txt"),
    ("How is a thread different from a process?", "os.txt"),
    ("Why do we split big tables into smaller ones?", "dbms.txt"),
    ("Which join keeps all rows from the left table?", "dbms.txt"),
    ("What does ACID stand for?", "dbms.txt"),
    ("Can I change a tuple after creating it?", "python.txt"),
    ("How do I look up a value by key in Python?", "python.txt"),
    ("Who won the cricket world cup?", None),        # not in the notes
    ("What is the capital of France?", None),         # not in the notes
]

# 1) Retrieval: is the right file among the top 3 chunks?
hits_ok = 0
answerable = [(q, f) for q, f in TEST_SET if f]
for q, expected in answerable:
    sources = [h["source"] for h in retrieve(q, 3)]
    ok = expected in sources
    hits_ok += ok
    print("OK  " if ok else "MISS", q, "->", sources)
print(f"retrieval hit rate@3: {hits_ok}/{len(answerable)}\n")

# 2) Refusal: does it correctly say "not in the notes"?
refused = 0
unanswerable = [q for q, f in TEST_SET if not f]
for q in unanswerable:
    r = answer(q)
    refused += not r["found"]
    print("refused" if not r["found"] else "ANSWERED", "|", q)
print(f"correct refusals: {refused}/{len(unanswerable)}")

OK   What is a deadlock? -> ['os.txt', 'dbms.txt', 'os.txt']
OK   What happens when two processes wait forever for each other's resources? -> ['os.txt', 'os.txt', 'dbms.txt']
OK   How is a thread different from a process? -> ['os.txt', 'os.txt', 'os.txt']
OK   Why do we split big tables into smaller ones? -> ['dbms.txt', 'os.txt', 'dbms.txt']
OK   Which join keeps all rows from the left table? -> ['dbms.txt', 'dbms.txt', 'dbms.txt']
OK   What does ACID stand for? -> ['dbms.txt', 'dbms.txt', 'os.txt']
OK   Can I change a tuple after creating it? -> ['python.txt', 'python.txt', 'dbms.txt']
OK   How do I look up a value by key in Python? -> ['python.txt', 'python.txt', 'dbms.txt']
retrieval hit rate@3: 8/8

refused | Who won the cricket world cup?
refused | What is the capital of France?
correct refusals: 2/2
